# 04 — Deduplicate

1. **Exact dedup**: hash whitespace-normalized lines. Where same hash appears in both
   GRETIL and DCS, keep the DCS copy (higher-quality annotation source). Log counts.
2. **Near-dup rate**: hash 5-gram bags; report rate but do NOT remove.
3. Output:
   - `data/clean/corpus.slp1.txt`  (one SLP1 line per unit)
   - `data/clean/index.csv`        (line_number, text_id, source, file)

In [1]:
import csv, hashlib, json, re
from collections import defaultdict
from pathlib import Path
from tqdm.auto import tqdm

BASE    = Path('/Users/sidharthbildikar/Desktop/code/llm-paninian-compression/sanskrit_corpus')
INTERIM = BASE / 'data' / 'interim'
CLEAN   = BASE / 'data' / 'clean'
CLEAN.mkdir(parents=True, exist_ok=True)

IN_JSONL    = INTERIM / 'corpus_slp1.jsonl'
OUT_TXT     = CLEAN / 'corpus.slp1.txt'
OUT_INDEX   = CLEAN / 'index.csv'

# Load all records
records = []
with IN_JSONL.open(encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))
print(f'Loaded {len(records):,} records')


Loaded 2,191,711 records


## Exact deduplication

Key = SHA-256 of whitespace-normalized line.
Rule: if a line appears in both DCS and GRETIL, keep DCS and drop GRETIL.

In [2]:
_WS = re.compile(r'\s+')

def norm_ws(text: str) -> str:
    return _WS.sub(' ', text).strip()

def line_hash(text: str) -> str:
    return hashlib.sha256(norm_ws(text).encode('utf-8')).hexdigest()

# First pass: build hash -> set of sources
hash_sources: dict[str, set] = defaultdict(set)
for rec in records:
    h = line_hash(rec['text'])
    hash_sources[h].add(rec['source'])

both_count    = sum(1 for s in hash_sources.values() if len(s) > 1)
print(f'Unique hashes          : {len(hash_sources):,}')
print(f'Hashes in BOTH sources : {both_count:,}  (will prefer DCS)')

# Second pass: deduplicate
seen_hashes: set = set()
kept: list       = []
dropped_dup  = 0
dropped_gretil_dcs_overlap = 0

# Process DCS first so DCS wins when same hash exists in both
dcs_recs     = [r for r in records if r['source'] == 'dcs']
gretil_recs  = [r for r in records if r['source'] == 'gretil']

for rec in dcs_recs:
    h = line_hash(rec['text'])
    if h not in seen_hashes:
        seen_hashes.add(h)
        kept.append(rec)
    else:
        dropped_dup += 1

for rec in gretil_recs:
    h = line_hash(rec['text'])
    if h not in seen_hashes:
        seen_hashes.add(h)
        kept.append(rec)
    else:
        # Check if this was a DCS/GRETIL overlap
        if 'dcs' in hash_sources[h]:
            dropped_gretil_dcs_overlap += 1
        else:
            dropped_dup += 1

print(f'\nDeduplication results:')
print(f'  Input lines             : {len(records):,}')
print(f'  Kept lines              : {len(kept):,}')
print(f'  Dropped (intra-source)  : {dropped_dup:,}')
print(f'  Dropped (GRETIL/DCS overlap, DCS preferred): {dropped_gretil_dcs_overlap:,}')
print(f'  Total dropped           : {len(records)-len(kept):,}')

Unique hashes          : 1,768,285
Hashes in BOTH sources : 263,338  (will prefer DCS)



Deduplication results:
  Input lines             : 2,191,711
  Kept lines              : 1,768,285
  Dropped (intra-source)  : 116,322
  Dropped (GRETIL/DCS overlap, DCS preferred): 307,104
  Total dropped           : 423,426


## Near-duplicate rate (5-gram hash bags, report only — do NOT remove)

In [3]:
from collections import Counter as _Counter
import random

def char_ngrams(text: str, n: int = 5) -> frozenset:
    """Return the set of character n-grams (deduplicated) in text."""
    normed = norm_ws(text)
    return frozenset(normed[i:i+n] for i in range(len(normed)-n+1))

def jaccard(a: frozenset, b: frozenset) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

# Estimate near-dup rate by LSH approximation:
# Hash each line's ngram set to a 64-bit minhash (single hash — cheap estimate).
# Two lines are 'near-dup' if their Jaccard estimate >= 0.8.
# We use a sampled comparison (10k random pairs) for speed.

NEAR_DUP_THRESHOLD = 0.8
PAIR_SAMPLE        = 10_000

random.seed(42)
sample_size = min(5000, len(kept))
sample_recs = random.sample(kept, sample_size)

# Build ngram sets for sample
ngram_sets = [char_ngrams(r['text']) for r in sample_recs]

# Random pair comparison
n_near = 0
n_pairs = min(PAIR_SAMPLE, sample_size * (sample_size - 1) // 2)
pairs_tested = 0

import random as _rand
idx_range = list(range(sample_size))
for _ in range(n_pairs):
    i, j = _rand.sample(idx_range, 2)
    if jaccard(ngram_sets[i], ngram_sets[j]) >= NEAR_DUP_THRESHOLD:
        n_near += 1
    pairs_tested += 1

near_dup_rate = n_near / pairs_tested if pairs_tested > 0 else 0
print(f'Near-duplicate analysis (5-gram Jaccard >= {NEAR_DUP_THRESHOLD}):')
print(f'  Sample lines     : {sample_size:,}')
print(f'  Pairs tested     : {pairs_tested:,}')
print(f'  Near-dup pairs   : {n_near:,}')
print(f'  Near-dup rate    : {100*near_dup_rate:.3f}% of pairs')
print(f'  (Near-dups are REPORTED ONLY — not removed)')

Near-duplicate analysis (5-gram Jaccard >= 0.8):
  Sample lines     : 5,000
  Pairs tested     : 10,000
  Near-dup pairs   : 0
  Near-dup rate    : 0.000% of pairs
  (Near-dups are REPORTED ONLY — not removed)


## Write final corpus + index

In [4]:
with OUT_TXT.open('w', encoding='utf-8') as ftxt, \
     OUT_INDEX.open('w', encoding='utf-8', newline='') as fcsv:

    writer = csv.writer(fcsv)
    writer.writerow(['line_number', 'text_id', 'source', 'file'])

    for line_num, rec in enumerate(kept, start=1):
        ftxt.write(rec['text'] + '\n')
        writer.writerow([line_num, rec['text_id'], rec['source'], rec['file']])

print(f'corpus.slp1.txt : {len(kept):,} lines -> {OUT_TXT}')
print(f'index.csv       : {len(kept):,} rows  -> {OUT_INDEX}')

# Persist stats for report notebook
dedup_stats = {
    'input_lines'               : len(records),
    'kept_lines'                : len(kept),
    'dropped_intra'             : dropped_dup,
    'dropped_gretil_dcs_overlap': dropped_gretil_dcs_overlap,
    'near_dup_rate_pct'         : round(100*near_dup_rate, 4),
    'near_dup_threshold'        : NEAR_DUP_THRESHOLD,
    'near_dup_pairs_tested'     : pairs_tested,
}
(INTERIM / 'dedup_stats.json').write_text(
    json.dumps(dedup_stats, indent=2), encoding='utf-8'
)
print('Dedup stats saved.')

corpus.slp1.txt : 1,768,285 lines -> /Users/sidharthbildikar/Desktop/code/llm-paninian-compression/sanskrit_corpus/data/clean/corpus.slp1.txt
index.csv       : 1,768,285 rows  -> /Users/sidharthbildikar/Desktop/code/llm-paninian-compression/sanskrit_corpus/data/clean/index.csv
Dedup stats saved.
